# Hassaniya ASR — Data Preparation

Builds the training dataset for the Whisper fine-tuning notebook (`hassanyia-whisper-medium.ipynb`).

**Steps:** load three public Hassaniya speech datasets → convert all audio to 16 kHz mono → merge → remove duplicates and non-speech clips → normalize the Arabic text → check Whisper length limits → group-aware 80/10/10 split → save parquet files + `manifest.json`.

**Runs locally** (CPU is enough). Set your folders in the **Paths** cell below before running.

## Setup
Imports used throughout the notebook.

In [1]:
import pandas as pd
import numpy as np
import soundfile as sf
import io

## Paths
The only machine-specific settings in this notebook — edit these to match your setup.

In [ ]:
# Folder holding the three downloaded datasets ("Dataset A", "Dataset B", "Dataset C" sub-folders)
DATA_ROOT = "path/to/Datasets"

# Local copy of a multilingual Whisper model (only its tokenizer is used; identical across Whisper sizes)
TOKENIZER_PATH = "path/to/whisper-small"

# Where the processed dataset folder (parquet files + manifest.json) is written
OUTPUT_ROOT = "path/to/output"

## Dataset A
Source: [Mamadou-Aw/hassaniya-parler-tts](https://huggingface.co/datasets/Mamadou-Aw/hassaniya-parler-tts) — 16 kHz mono. Train and test files are merged; only `audio` and `text` are kept.

In [2]:
train = pd.read_parquet(f"{DATA_ROOT}/Dataset A/train-0001.parquet")
test = pd.read_parquet(f"{DATA_ROOT}/Dataset A/test-0001.parquet")

In [3]:
#merge train and test dataset
datasetA = pd.concat([train, test], ignore_index=True)
datasetA.head()

,audio,old_transcription,translation,text,text_description
0,{'bytes': b'RIFFds\x01\x00WAVEfmt \x10\x00\x00...,مشكلة الموريتانيين النفاق الاجتماعي والنفاق لل...,Le problème des Mauritaniens est l'hypocrisie ...,مشكلة الموريتانيين النفاق الاجتماعي والنفاق لل...,A single speaker delivers clear speech at a mo...
1,{'bytes': b'RIFF\x84\x0b\x01\x00WAVEfmt \x10\x...,إطلس و إيحاول لاتلاو يدخل شبه,Atlas et il essaie de ne pas entrer dans la semi.,إطلس و إيحاول لاتلاو يدخل شبه,A single speaker delivers clear speech at a mo...
2,{'bytes': b'RIFF\xc4\x83\x01\x00WAVEfmt \x10\x...,أكثر ركاج في سوشن ميديا يتقاضي أعلي ليعلانات,La plupart des influenceurs sur les réseaux so...,أكثر ركاج في سوشن ميديا يتقاضي أعلي ليعلانات,A single speaker delivers clear speech at a mo...
3,{'bytes': b'RIFF\xe4\x85\x01\x00WAVEfmt \x10\x...,نركب وتي ويسولني مولاه كان عندي كارداليت,Je monte et il me demande mon maître si j'avai...,نركب وتي ويسولني مولاه كان عندي كارداليت,A single speaker delivers clear speech at a mo...
4,{'bytes': b'RIFF$\xa4\x00\x00WAVEfmt \x10\x00\...,والل شي يسو شنه,"Et bien, que peut-on faire ?",والل شي يسو شنه,A single speaker delivers clear speech at a mo...


In [4]:
datasetA.columns

Index(['audio', 'old_transcription', 'translation', 'text',
       'text_description'],
      dtype='str')

In [5]:
datasetA.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   audio              1000 non-null   object
 1   old_transcription  1000 non-null   str   
 2   translation        1000 non-null   str   
 3   text               1000 non-null   str   
 4   text_description   1000 non-null   str   
dtypes: object(1), str(4)
memory usage: 337.5+ KB


In [6]:
datasetA.iloc[0]

audio                {'bytes': b'RIFFds\x01\x00WAVEfmt \x10\x00\x00...
old_transcription    مشكلة الموريتانيين النفاق الاجتماعي والنفاق لل...
translation          Le problème des Mauritaniens est l'hypocrisie ...
text                 مشكلة الموريتانيين النفاق الاجتماعي والنفاق لل...
text_description     A single speaker delivers clear speech at a mo...
Name: 0, dtype: object

In [7]:
#sample rate, shape duration
audio_bytes = datasetA.iloc[0]["audio"]["bytes"]

data, samplerate = sf.read(io.BytesIO(audio_bytes))

print("Sample rate:", samplerate)
print("Shape:", data.shape)
print("Duration (seconds):", len(data) / samplerate)

Sample rate: 16000
Shape: (47520,)
Duration (seconds): 2.97


In [8]:
#are all recordings same rate
sample_rates = set()

for i in range(len(datasetA)):
    audio_bytes = datasetA.iloc[i]["audio"]["bytes"]
    data, sr = sf.read(io.BytesIO(audio_bytes))
    sample_rates.add(sr)

print(sample_rates)

{16000}


In [9]:
#records length
durations = []

for i in range(len(datasetA)):
    audio_bytes = datasetA.iloc[i]["audio"]["bytes"]
    data, sr = sf.read(io.BytesIO(audio_bytes))
    durations.append(len(data) / sr)

print("Shortest:", min(durations))
print("Longest :", max(durations))
print("Average :", sum(durations)/len(durations))

Shortest: 0.456
Longest : 17.159
Average : 2.3604712500000002


In [10]:
#any empty transcripts or duplicate
print("empty: " , (datasetA["text"].str.strip() == "").sum())
print("duplicate: " , datasetA["text"].duplicated().sum())

empty:  0
duplicate:  3


In [11]:
datasetA = datasetA.drop(columns=["old_transcription" , "translation" , "text_description"])
datasetA["source"] = "datasetA"
datasetA.columns

Index(['audio', 'text', 'source'], dtype='str')

## Dataset B
Source: [hadramibegnoug/abc.hassaniya_ASR](https://huggingface.co/datasets/hadramibegnoug/abc.hassaniya_ASR) — 48 kHz stereo, converted to 16 kHz mono below.

In [12]:
train1 = pd.read_parquet(f"{DATA_ROOT}/Dataset B/test-00000-of-00002.parquet")
train2 = pd.read_parquet(f"{DATA_ROOT}/Dataset B/validation-00000-of-00002.parquet")
train3 = pd.read_parquet(f"{DATA_ROOT}/Dataset B/test-00001-of-00002.parquet")
train4 = pd.read_parquet(f"{DATA_ROOT}/Dataset B/validation-00001-of-00002.parquet")

datasetB = pd.concat([train1, train2, train3, train4], ignore_index=True)

In [13]:
datasetB.head()

,audio,seg_id,transcription,gender,duration
0,{'bytes': b'RIFFtK\x07\x00WAVEfmt \x10\x00\x00...,59356_2,احنا مانه لي نفرصو في الناس ذي القضية ندوروا ن...,Missing,2.489744
1,{'bytes': b'RIFF\xa8\xa4\x0c\x00WAVEfmt \x10\x...,59357_5,ولين تخلك فظتنا الي قادية تنجبر نحلوا المشكلة ...,Missing,4.315353
2,{'bytes': b'RIFF4D\t\x00WAVEfmt \x10\x00\x00\x...,59358_1,صافا اياك لباس اياك الخير,Missing,3.162759
3,{'bytes': b'RIFF\xd4\xb0\x02\x00WAVEfmt \x10\x...,59359_2,ايو شلاهي تواسي,Missing,0.918260
4,{'bytes': b'RIFF\x94\x1a\n\x00WAVEfmt \x10\x00...,59359_3,أنا ذي السنة عندي باك تعرف ولاني صايب,Missing,3.448575


In [14]:
datasetB.columns

Index(['audio', 'seg_id', 'transcription', 'gender', 'duration'], dtype='str')

In [15]:
datasetB.info()

<class 'pandas.DataFrame'>
RangeIndex: 1906 entries, 0 to 1905
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   audio          1906 non-null   object 
 1   seg_id         1906 non-null   str    
 2   transcription  1906 non-null   str    
 3   gender         1898 non-null   str    
 4   duration       1906 non-null   float64
dtypes: float64(1), object(1), str(3)
memory usage: 256.0+ KB


In [16]:
datasetB.iloc[0]

audio            {'bytes': b'RIFFtK\x07\x00WAVEfmt \x10\x00\x00...
seg_id                                                     59356_2
transcription    احنا مانه لي نفرصو في الناس ذي القضية ندوروا ن...
gender                                                     Missing
duration                                                  2.489744
Name: 0, dtype: object

In [17]:
#sample rate, shape duration
audio_bytes = datasetB.iloc[0]["audio"]["bytes"]

data, samplerate = sf.read(io.BytesIO(audio_bytes))

print("Sample rate:", samplerate)
print("Shape:", data.shape)
print("Duration (seconds):", len(data) / samplerate)

Sample rate: 48000
Shape: (119508, 2)
Duration (seconds): 2.48975


In [26]:
#are all recordings same rate
sample_rates = set()

for i in range(len(datasetB)):
    audio_bytes = datasetB.iloc[i]["audio"]["bytes"]
    data, sr = sf.read(io.BytesIO(audio_bytes))
    sample_rates.add(sr)

print(sample_rates)

{16000}


In [19]:
#records length
durations = []

for i in range(len(datasetB)):
    audio_bytes = datasetB.iloc[i]["audio"]["bytes"]
    data, sr = sf.read(io.BytesIO(audio_bytes))
    durations.append(len(data) / sr)

print("Shortest:", min(durations))
print("Longest :", max(durations))
print("Average :", sum(durations)/len(durations))

Shortest: 0.6199375
Longest : 27.075
Average : 3.6820070391745365


In [20]:
#any empty transcripts or duplicate
print("empty: " , (datasetB["transcription"].str.strip() == "").sum())
print("duplicate: " , datasetB["transcription"].duplicated().sum())

empty:  0
duplicate:  16


In [21]:
datasetB = datasetB[["audio", "transcription"]].copy()

datasetB = datasetB.rename(columns={
    "transcription": "text"
})

datasetB["source"] = "datasetB"
datasetB.columns

Index(['audio', 'text', 'source'], dtype='str')

### Convert to 16 kHz mono
Whisper expects 16 kHz mono audio; Dataset B is 48 kHz stereo.

In [22]:
import io
import soundfile as sf
import librosa

In [23]:
# conversion function
def convert_to_16k_mono(audio_dict):
    # Get the original WAV bytes
    audio_bytes = audio_dict["bytes"]

    # Read the audio
    audio, sr = sf.read(io.BytesIO(audio_bytes))

    # Convert stereo to mono if necessary
    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    # Resample to 16 kHz if necessary
    if sr != 16000:
        audio = librosa.resample(
            audio,
            orig_sr=sr,
            target_sr=16000
        )

    # Save the processed audio back into WAV bytes
    output = io.BytesIO()

    sf.write(
        output,
        audio,
        16000,
        format="WAV"
    )

    return {
        "bytes": output.getvalue(),
        "path": audio_dict.get("path")
    }


In [24]:
datasetB["audio"] = datasetB["audio"].apply(convert_to_16k_mono)

C:\Users\haaat\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [25]:
audio_bytes = datasetB.iloc[0]["audio"]["bytes"]

data, sr = sf.read(io.BytesIO(audio_bytes))

print("Sample rate:", sr)
print("Shape:", data.shape)
print("Duration:", len(data) / sr)

Sample rate: 16000
Shape: (39836,)
Duration: 2.48975


In [27]:
sample_rates = set()
channels = set()

for i in range(len(datasetB)):
    audio_bytes = datasetB.iloc[i]["audio"]["bytes"]
    audio, sr = sf.read(io.BytesIO(audio_bytes))

    sample_rates.add(sr)

    if audio.ndim == 1:
        channels.add(1)
    else:
        channels.add(audio.shape[1])

print("Sample rates:", sample_rates)
print("Channels:", channels)

Sample rates: {16000}
Channels: {1}


## Dataset C
Source: [Mamadou-Aw/Hassaniya-speech-dataset](https://huggingface.co/datasets/Mamadou-Aw/Hassaniya-speech-dataset) — 16 kHz mono.

In [28]:
datasetC = pd.read_parquet(f"{DATA_ROOT}/Dataset C/audios_dataset.parquet")
datasetC.head(15)

,audio,transcription
0,{'bytes': b'RIFF$\x9a\x00\x00WAVEfmt \x10\x00\...,السلام عليكم ورحمة الله
1,{'bytes': b'RIFF\xe4{\x01\x00WAVEfmt \x10\x00\...,ويدبزوه فماتياه بعد قاع لين تساوة كرامة بتراب
2,{'bytes': b'RIFF\xe4Z\x00\x00WAVEfmt \x10\x00\...,في كيهيدي
3,{'bytes': b'RIFFD\xaa\x01\x00WAVEfmt \x10\x00\...,أتفيق الحكومة في دقيقة وحده وذهو الوالي منكز
4,{'bytes': b'RIFF\x84O\x01\x00WAVEfmt \x10\x00\...,الوالي منكز أعلاش مواطنين أكتل كان متظاهرين
5,{'bytes': b'RIFF\x84\xb5\x00\x00WAVEfmt \x10\x...,مواطنين مات في الحبس
6,{'bytes': b'RIFF$o\x01\x00WAVEfmt \x10\x00\x00...,ميزانيتك أكثر من أحطعشر مليار ماقط دارت كرياج ...
7,{'bytes': b'RIFF\xa4\x05\x01\x00WAVEfmt \x10\x...,لماذا ماتم أقالت وزير الشؤون المشوية
8,{'bytes': b'RIFF\xa4|\x00\x00WAVEfmt \x10\x00\...,ولين أموت أملي
9,{'bytes': b'RIFF\xa4\xc5\x01\x00WAVEfmt \x10\x...,لماذا مانسمع أقالات في الجملة فهذي الوزارة الذ...


In [29]:
datasetC.info()

<class 'pandas.DataFrame'>
RangeIndex: 294 entries, 0 to 293
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   audio          294 non-null    object
 1   transcription  294 non-null    str   
dtypes: object(1), str(1)
memory usage: 21.9+ KB


In [30]:
#sample rate, shape duration
audio_bytes = datasetC.iloc[0]["audio"]["bytes"]

data, samplerate = sf.read(io.BytesIO(audio_bytes))

print("Sample rate:", samplerate)
print("Shape:", data.shape)
print("Duration (seconds):", len(data) / samplerate)

Sample rate: 16000
Shape: (19712,)
Duration (seconds): 1.232


In [31]:
#are all recordings same rate
sample_rates = set()

for i in range(len(datasetC)):
    audio_bytes = datasetC.iloc[i]["audio"]["bytes"]
    data, sr = sf.read(io.BytesIO(audio_bytes))
    sample_rates.add(sr)

print(sample_rates)

{16000}


In [32]:
#records length
durations = []

for i in range(len(datasetC)):
    audio_bytes = datasetC.iloc[i]["audio"]["bytes"]
    data, sr = sf.read(io.BytesIO(audio_bytes))
    durations.append(len(data) / sr)

print("Shortest:", min(durations))
print("Longest :", max(durations))
print("Average :", sum(durations)/len(durations))

Shortest: 0.506
Longest : 9.77
Average : 2.118359906462585


In [33]:
#any empty transcripts or duplicate
print("empty: " , (datasetC["transcription"].str.strip() == "").sum())
print("duplicate: " , datasetC["transcription"].duplicated().sum())

empty:  0
duplicate:  0


In [34]:
datasetC = datasetC[["audio", "transcription"]].copy()

datasetC = datasetC.rename(columns={
    "transcription": "text"
})

datasetC["source"] = "datasetC"
datasetC.columns

Index(['audio', 'text', 'source'], dtype='str')

## Merge the datasets
All three now share the columns `audio`, `text`, `source`.

In [35]:
print(datasetA.columns)
print(datasetB.columns)
print(datasetC.columns)

Index(['audio', 'text', 'source'], dtype='str')
Index(['audio', 'text', 'source'], dtype='str')
Index(['audio', 'text', 'source'], dtype='str')


In [36]:
combined = pd.concat(
    [datasetA, datasetB, datasetC],
    ignore_index=True
)

print("Total recordings:", len(combined))
print(combined.columns)

Total recordings: 3200
Index(['audio', 'text', 'source'], dtype='str')


In [37]:
print(combined["source"].value_counts())

source
datasetB    1906
datasetA    1000
datasetC     294
Name: count, dtype: int64


### Check for duplicate audio
Identical recordings are found by hashing the raw WAV bytes.

In [38]:
import hashlib

def audio_hash(audio_dict):
    return hashlib.md5(audio_dict["bytes"]).hexdigest()

combined["audio_hash"] = combined["audio"].apply(audio_hash)

In [39]:
# count duplicates
duplicate_audio = combined["audio_hash"].duplicated().sum()

print("Duplicate audio recordings:", duplicate_audio)

Duplicate audio recordings: 52


In [40]:
# detect duplicates
duplicates = combined[combined["audio_hash"].duplicated(keep=False)]

duplicates[["text", "source", "audio_hash"]].sort_values("audio_hash")

,text,source,audio_hash
491,السلام عليكم ورحمة الله,datasetA,0197002b0ffdaaf129638d32e8252e36
945,مهتمين بافريقيا ومهتمين بالعالم الإسلامي دوائر...,datasetA,0197002b0ffdaaf129638d32e8252e36
212,ولا عندنا صنم,datasetA,01efd6f4faae8fee306e509c4d4d85e7
363,أراعيهم ذا ليكولو اللا الاساسيات معناها ارواية...,datasetA,01efd6f4faae8fee306e509c4d4d85e7
642,اليمينية المتطرفة حتى عنها ما اكتفات,datasetA,0badf367e9b429340180f3cda34865de
...,...,...,...
376,أنا عدلت كوليج فبوحديدة,datasetA,ef2b31d64892438e005b006567959dc4
211,لاهي نتكلمو عن,datasetA,f0dba28afc466023451930857f028344
831,وحاجلي هوم تعرف ذا التكوين فيه محاضرات وفيه شي...,datasetA,f0dba28afc466023451930857f028344
71,لبنتك فيها مية جنيه لاهي تعطيها لمنادم اللي لا...,datasetA,f34eebdcea57d48e5a24175429722044


The duplicated recordings carry **different transcripts**, so neither copy can be trusted.

In [41]:
duplicates.groupby("audio_hash")["source"].unique()

audio_hash
0197002b0ffdaaf129638d32e8252e36    [datasetA]
01efd6f4faae8fee306e509c4d4d85e7    [datasetA]
0badf367e9b429340180f3cda34865de    [datasetA]
0d0b8c2044fd4ec8b199837a90ccfa1c    [datasetA]
112237e112806e8d3d507e8a0d4aee0c    [datasetA]
13bfb455e1b35128499126d158f51d7c    [datasetA]
1e2cfbef5a154e45f09b3453a8b6d1be    [datasetA]
2d5b46db8573c49777cae0d81056c0f4    [datasetA]
33a82167c8e06d95a4e0e3ce317b5717    [datasetA]
33b34855d599f4eb418202f872d49c2c    [datasetA]
35aa9bbf47ed5de343aa67b54066a2b1    [datasetA]
35b4b2eeea09c27b82436b4c0903021c    [datasetA]
377aded28098ac7aed9b641cac76ed67    [datasetA]
45b5aff544d1f31cbbe3d13c148a62d2    [datasetA]
4798b6ad9c6dcc02d6de57ec806b4db7    [datasetA]
4fa23e4662792d85408ce4012b7faeee    [datasetA]
5330d2ecf22c50f5627868fe4b0a4233    [datasetA]
544a38f494c7c6310787dda473e0cfb6    [datasetA]
558fb997ee3f16da48a70ee6351256a9    [datasetA]
5eb27c8ceba89b1595676f105dbb7ab1    [datasetA]
63a3e865c814cb8174bb734a64467041    [datasetA]
65

In [42]:
print("Unique duplicated audio:", duplicates["audio_hash"].nunique())

Unique duplicated audio: 52


In [43]:
# check duplicate audios with different transcript
duplicates.groupby("audio_hash")["text"].nunique().sort_values(ascending=False).head(20)

audio_hash
0197002b0ffdaaf129638d32e8252e36    2
01efd6f4faae8fee306e509c4d4d85e7    2
0badf367e9b429340180f3cda34865de    2
0d0b8c2044fd4ec8b199837a90ccfa1c    2
112237e112806e8d3d507e8a0d4aee0c    2
13bfb455e1b35128499126d158f51d7c    2
1e2cfbef5a154e45f09b3453a8b6d1be    2
2d5b46db8573c49777cae0d81056c0f4    2
33a82167c8e06d95a4e0e3ce317b5717    2
33b34855d599f4eb418202f872d49c2c    2
35aa9bbf47ed5de343aa67b54066a2b1    2
35b4b2eeea09c27b82436b4c0903021c    2
377aded28098ac7aed9b641cac76ed67    2
45b5aff544d1f31cbbe3d13c148a62d2    2
4798b6ad9c6dcc02d6de57ec806b4db7    2
4fa23e4662792d85408ce4012b7faeee    2
5330d2ecf22c50f5627868fe4b0a4233    2
544a38f494c7c6310787dda473e0cfb6    2
558fb997ee3f16da48a70ee6351256a9    2
5eb27c8ceba89b1595676f105dbb7ab1    2
Name: text, dtype: int64

### Remove duplicates
Every copy of a duplicated recording is dropped.

In [44]:
duplicate_hashes = (
    combined["audio_hash"]
    .value_counts()
)

duplicate_hashes = duplicate_hashes[
    duplicate_hashes > 1
].index

In [45]:
combined_clean = combined[
    ~combined["audio_hash"].isin(duplicate_hashes)
].copy()

In [46]:
print("Before:", len(combined))
print("After:", len(combined_clean))
print("Removed:", len(combined) - len(combined_clean))

Before: 3200
After: 3096
Removed: 104


In [47]:
print(
    "Remaining duplicate audio:",
    combined_clean["audio_hash"].duplicated().sum()
)

Remaining duplicate audio: 0


## Clean the transcripts
Check for missing/empty text and strip stray whitespace.

In [48]:
# missing and empty transcripts
print("Missing transcripts:", combined_clean["text"].isna().sum())

empty_text = (
    combined_clean["text"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print("Empty transcripts:", empty_text)

Missing transcripts: 0
Empty transcripts: 0


In [49]:
# check transcript length
text_lengths = combined_clean["text"].astype(str).str.len()

print("Shortest transcript:", text_lengths.min())
print("Longest transcript:", text_lengths.max())
print("Average transcript length:", text_lengths.mean())

Shortest transcript: 4
Longest transcript: 391
Average transcript length: 43.73320413436692


In [50]:
# whitespace issues
whitespace_issues = combined_clean[
    combined_clean["text"].astype(str).str.match(r"^\s|\s$")
]

print("Transcripts with leading/trailing whitespace:",
      len(whitespace_issues))

Transcripts with leading/trailing whitespace: 13


In [51]:
combined_clean["text"] = combined_clean["text"].str.strip()

### Remove non-speech clips
Drop clips whose transcript is only a tag such as `[singing]` or `[crying]`.

In [52]:
# remove rows whose transcript is a non-speech annotation tag,
# e.g. "[singing]", "[crying]", "[laughing]" (and typo variants like "[singinig]")
non_speech_mask = combined_clean["text"].str.contains(
    r"\[[^\]]*\]", regex=True, na=False
)

print("Rows with non-speech annotation tags:", non_speech_mask.sum())

combined_clean = combined_clean[~non_speech_mask].copy()

Rows with non-speech annotation tags: 14


### Normalize Arabic text

The same `normalize_arabic()` is applied to the training labels here **and** to both references and predictions at evaluation time (the training notebook contains an identical copy — keep the two in sync). The model therefore trains on the same text distribution it is graded on.

Steps: Unicode NFKC (folds Arabic presentation forms), strip diacritics, fold alef variants (أ/إ/آ/ٱ → ا) and alef maqsura (ى → ي), remove tatweel, replace every non-word character (Arabic and Latin punctuation alike) with a space, collapse whitespace. Hassaniya letters such as گ are kept.

In [53]:
import re
import unicodedata

def normalize_arabic(text):
    if text is None:
        return ""

    text = unicodedata.normalize("NFKC", str(text))

    # Remove Arabic diacritics
    text = re.sub(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]", "", text)

    # Normalize alef variants
    text = re.sub(r"[أإآٱ]", "ا", text)

    # Alef maqsura -> ya
    text = text.replace("ى", "ي")

    # Remove tatweel
    text = text.replace("ـ", "")

    # Remove punctuation (Arabic ، ؛ ؟ included; letters, digits and gaf stay)
    text = re.sub(r"[^\w\s]|_", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


# a missing transcript would otherwise become the string "nan"
combined_clean = combined_clean.dropna(subset=["text"])
combined_clean["text"] = combined_clean["text"].apply(normalize_arabic)

# Normalization can empty a transcript that was only punctuation
empty_after_norm = combined_clean["text"].eq("")
print("Empty after normalization (dropped):", empty_after_norm.sum())
combined_clean = combined_clean[~empty_after_norm].copy()

print(combined_clean["text"].head(10))

Empty after normalization (dropped): 0
0    مشكلة الموريتانيين النفاق الاجتماعي والنفاق لل...
1                        اطلس و ايحاول لاتلاو يدخل شبه
2         اكثر ركاج في سوشن ميديا يتقاضي اعلي ليعلانات
3             نركب وتي ويسولني مولاه كان عندي كارداليت
4                                      والل شي يسو شنه
5                                 جات ملي لكلام لها هي
6    گالو القاضي هي گالت انك مشيت بيها شور دار وطرشتها
7     ويلين وتلفن لي ذيك الساعة گالو علي ذي القضية جيت
8                                   اكلاس رايد هي الحل
9    انت گالو اص گاع اخبارك من باسكنو انت ماعندك في...
Name: text, dtype: str


In [54]:
print(combined_clean["source"].value_counts())

source
datasetB    1892
datasetA     896
datasetC     294
Name: count, dtype: int64


### Length checks (Whisper limits)

Whisper takes at most 30 s of audio and 448 decoder tokens. Labels are tokenized with the language/task prefix (`<|startoftranscript|><|ar|><|transcribe|><|notimestamps|>`), exactly as the training script will do, so the token count here is the real one. The multilingual tokenizer is identical across Whisper sizes, so the local whisper-small copy gives the same ids as whisper-medium.

In [55]:
from transformers import WhisperTokenizer

MAX_AUDIO_SECONDS = 30.0
MAX_LABEL_TOKENS = 448

tokenizer = WhisperTokenizer.from_pretrained(
    TOKENIZER_PATH,
    language="arabic",
    task="transcribe",
    local_files_only=True,
)

# Labels must start with the full Whisper prefix, or training and inference prompts disagree
expected_prefix = tokenizer.convert_tokens_to_ids(
    ["<|startoftranscript|>", "<|ar|>", "<|transcribe|>", "<|notimestamps|>"]
)
sample_ids = tokenizer(combined_clean["text"].iloc[0]).input_ids
assert sample_ids[:4] == expected_prefix, (sample_ids[:4], expected_prefix)
print("Label prefix OK:", tokenizer.convert_ids_to_tokens(sample_ids[:4]))

combined_clean["duration"] = [
    sf.info(io.BytesIO(audio["bytes"])).duration
    for audio in combined_clean["audio"]
]
combined_clean["n_tokens"] = [
    len(tokenizer(text).input_ids) for text in combined_clean["text"]
]

too_long = (
    (combined_clean["duration"] > MAX_AUDIO_SECONDS)
    | (combined_clean["n_tokens"] > MAX_LABEL_TOKENS)
)
print("Longest audio (s):", round(combined_clean["duration"].max(), 2))
print("Longest label (tokens):", combined_clean["n_tokens"].max())
print("Rows over Whisper limits (dropped):", too_long.sum())

combined_clean = combined_clean[~too_long].drop(columns=["n_tokens"]).copy()
print("Rows:", len(combined_clean))

Label prefix OK: ['<|startoftranscript|>', '<|ar|>', '<|transcribe|>', '<|notimestamps|>']
Longest audio (s): 27.08
Longest label (tokens): 223
Rows over Whisper limits (dropped): 0
Rows: 3082


### Shuffle
Fixed seed (42) so the split is reproducible.

In [56]:
# shuffle the dataset
combined_clean = combined_clean.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

In [57]:
print(combined_clean["source"].head(20))

0     datasetB
1     datasetC
2     datasetA
3     datasetA
4     datasetB
5     datasetA
6     datasetA
7     datasetB
8     datasetC
9     datasetA
10    datasetA
11    datasetA
12    datasetC
13    datasetB
14    datasetA
15    datasetB
16    datasetB
17    datasetB
18    datasetB
19    datasetB
Name: source, dtype: str


## Split into train / validation / test (80 / 10 / 10)

### Group-aware split (prevents same-episode segments from leaking across train/val/test)

Dataset B's filenames encode the source episode/show title before the clip's start/end timestamps and segment id (`<episode title>_<start>_<end>_<seg_id>.wav`, where the seg_id itself looks like `59356_2`). Segments from the same episode share speakers and acoustic conditions, so a plain random split can put some of an episode's segments in train and others in test/val, inflating apparent performance. Rows are grouped by episode for Dataset B; Dataset A and C have no reliable shared-recording identifier (A's only signal is a recording date shared by hundreds of rows, far too coarse to use without badly skewing the 80/10/10 split), so each of their rows is treated as its own independent group.

In [58]:
episode_pattern = re.compile(r"^(.*)_[\d.]+_[\d.]+_\d+_\d+\.wav$")

def get_group_id(source, path):
    if source == "datasetB" and path:
        match = episode_pattern.match(path)
        if match:
            return f"datasetB::{match.group(1)}"
    return None

combined_clean["group_id"] = [
    get_group_id(src, audio.get("path"))
    for src, audio in zip(combined_clean["source"], combined_clean["audio"])
]

no_group = combined_clean["group_id"].isna()
combined_clean.loc[no_group, "group_id"] = (
    "row::" + combined_clean.index[no_group].astype(str)
)

print("Unique groups:", combined_clean["group_id"].nunique())
print("Rows:", len(combined_clean))

Unique groups: 1399
Rows: 3082


In [59]:
from sklearn.model_selection import GroupShuffleSplit

split1 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, temp_idx = next(
    split1.split(combined_clean, groups=combined_clean["group_id"])
)
train_df = combined_clean.iloc[train_idx]
temp_df = combined_clean.iloc[temp_idx]

split2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(
    split2.split(temp_df, groups=temp_df["group_id"])
)
val_df = temp_df.iloc[val_idx]
test_df = temp_df.iloc[test_idx]

In [60]:
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 2525
Validation: 268
Test: 289


In [61]:
print("\nTrain Dataset(80%):")
print(train_df["source"].value_counts())

print("\nValidation Dataset(10%):")
print(val_df["source"].value_counts())

print("\nTest Dataset(10%):")
print(test_df["source"].value_counts())


Train Dataset(80%):
source
datasetB    1579
datasetA     702
datasetC     244
Name: count, dtype: int64

Validation Dataset(10%):
source
datasetB    145
datasetA     92
datasetC     31
Name: count, dtype: int64

Test Dataset(10%):
source
datasetB    168
datasetA    102
datasetC     19
Name: count, dtype: int64


## Save the dataset

The saved files hold **16 kHz mono WAV bytes + normalized text**, not precomputed Whisper features. Feature extraction and tokenization happen in the training script, which keeps the upload ~10× smaller and works for any Whisper size (large-v3 needs 128 mel bins, not 80).

A `manifest.json` is written next to the parquet files (dataset version, seed, row counts per split and source, total audio hours, MD5 per file). Upload the whole folder to Kaggle; the training script prints the manifest so every run records exactly which data it used.

In [62]:
import os
import json
import hashlib
from datetime import datetime, timezone

DATASET_VERSION = "v2"
SPLIT_SEED = 42
OUTPUT_DIR = f"{OUTPUT_ROOT}/hassaniya-asr-{DATASET_VERSION}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SAVE_COLUMNS = ["audio", "text", "source", "group_id", "duration", "audio_hash"]

def file_md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = {
    "dataset_version": DATASET_VERSION,
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "split_seed": SPLIT_SEED,
    "split_method": "GroupShuffleSplit 80/10/10 (Dataset B grouped by episode)",
    "audio_format": "WAV bytes, 16 kHz mono",
    "text_normalization": "normalize_arabic (NFKC, diacritics, alef/ya folding, tatweel, punctuation)",
    "splits": {},
}

for split_name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    path = f"{OUTPUT_DIR}/hassaniya_{split_name}.parquet"
    df[SAVE_COLUMNS].reset_index(drop=True).to_parquet(path, index=False)

    manifest["splits"][split_name] = {
        "file": os.path.basename(path),
        "rows": len(df),
        "hours": round(df["duration"].sum() / 3600, 3),
        "rows_per_source": df["source"].value_counts().sort_index().to_dict(),
        "md5": file_md5(path),
    }

with open(f"{OUTPUT_DIR}/manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("Saved to:", OUTPUT_DIR)
print(json.dumps(manifest["splits"], indent=2))

Saved to: C:/Users/haaat/Desktop/Hassaniya/Created Dataset/hassaniya-asr-v2
{
  "train": {
    "file": "hassaniya_train.parquet",
    "rows": 2525,
    "hours": 2.227,
    "rows_per_source": {
      "datasetA": 702,
      "datasetB": 1579,
      "datasetC": 244
    },
    "md5": "c716c5c03ff3a09217c5f49ac452eab2"
  },
  "validation": {
    "file": "hassaniya_validation.parquet",
    "rows": 268,
    "hours": 0.221,
    "rows_per_source": {
      "datasetA": 92,
      "datasetB": 145,
      "datasetC": 31
    },
    "md5": "9b45c6d04faf23708ba93c17e65f903c"
  },
  "test": {
    "file": "hassaniya_test.parquet",
    "rows": 289,
    "hours": 0.234,
    "rows_per_source": {
      "datasetA": 102,
      "datasetB": 168,
      "datasetC": 19
    },
    "md5": "a66bdefd44672de97b2234ebcafe27ff"
  }
}


In [63]:
# Round-trip check: reload the saved files and make sure nothing leaked or got lost
reloaded = {
    name: pd.read_parquet(f"{OUTPUT_DIR}/hassaniya_{name}.parquet")
    for name in ["train", "validation", "test"]
}

for name, df in reloaded.items():
    assert len(df) == manifest["splits"][name]["rows"], name
    assert df["text"].str.len().gt(0).all(), name
    audio, sr = sf.read(io.BytesIO(df.iloc[0]["audio"]["bytes"]))
    assert sr == 16000 and audio.ndim == 1, (name, sr, audio.shape)

train_groups = set(reloaded["train"]["group_id"])
val_groups = set(reloaded["validation"]["group_id"])
test_groups = set(reloaded["test"]["group_id"])
assert not (train_groups & val_groups), "group leak train/val"
assert not (train_groups & test_groups), "group leak train/test"
assert not (val_groups & test_groups), "group leak val/test"

all_hashes = pd.concat([df["audio_hash"] for df in reloaded.values()])
assert not all_hashes.duplicated().any(), "duplicate audio across splits"

print("Saved dataset verified: row counts, 16 kHz mono audio, no group or audio overlap.")

Saved dataset verified: row counts, 16 kHz mono audio, no group or audio overlap.
